### Structured output

Structured output allows agents to return data in a specific, predictable format. Instead of parsing natural language responses, you get structured data in the form of JSON objects, Pydantic models, or dataclasses that your application can use directly.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [9]:

import os
from dotenv import load_dotenv

load_dotenv()

# os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
os.environ['GEMINI_API_KEY']=os.getenv('GEMINI_API_KEY')

In [10]:
import os
from langchain.chat_models import init_chat_model
# model = init_chat_model("groq:qwen/qwen3.6-27b")
model = init_chat_model("groq:openai/gpt-oss-120b")
response=model.invoke("Hellow, how are you?")
print(response)

content="Hello! I'm doing great, thank you for asking. How can I help you today?" additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT. The user says "Hellow, how are you?" It\'s a greeting. We respond politely.'} response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 78, 'total_tokens': 133, 'completion_time': 0.113672631, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.002866286, 'prompt_tokens_details': None, 'queue_time': 0.306187788, 'total_time': 0.116538917}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c800245357', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a077bc-d8ae-7250-9ba6-2305711cf89c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 78, 'output_tokens': 55, 'total_tokens': 133, 'output_token_details': {'reasoning': 28}}


In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie.")
    year: int = Field(..., description="The release year of the movie.")
    director: str = Field(..., description="The director of the movie.")
    rating: float = Field(..., description="The rating of the movie.")


In [12]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure


_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021A4EFF9950>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021A4EFFA350>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The t

In [13]:
model_with_structure.invoke("Provide details of the movie Titanic")

Movie(title='Titanic', year=1997, director='James Cameron', rating=7.8)

#### Message output alongside past structure

In [15]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie.")
    year: int = Field(..., description="The release year of the movie.")
    director: str = Field(..., description="The director of the movie.")
    rating: float = Field(..., description="The rating of the movie.")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide details of the movie Titanic")
response



{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants details of the movie Titanic. We need to provide details: director, rating, title, year. Use the function Movie. Probably we need to call the function with appropriate data. Titanic: director James Cameron, rating maybe 7.8? IMDb rating 7.8. Year 1997. Title "Titanic". Provide function call.', 'tool_calls': [{'id': 'fc_4cb80961-e5b1-49ca-a560-2d0e5d2749b2', 'function': {'arguments': '{"director":"James Cameron","rating":7.8,"title":"Titanic","year":1997}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 125, 'prompt_tokens': 155, 'total_tokens': 280, 'completion_time': 0.260113793, 'completion_tokens_details': {'reasoning_tokens': 73}, 'prompt_time': 0.005802536, 'prompt_tokens_details': None, 'queue_time': 0.213255653, 'total_time': 0.265916329}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5d041d5b40', 'service_tier': 'on_demand', 'finis

#### Nested structure

In [ ]:
from pydnatic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    generes: list[str]
    budget: float | None = Field(None, description="The budget of the movie in million USD.")

model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)
response = model_with_structure.invoke("Provide details of the movie Titanic")
response